# Outlier removal with SOR

This notebook shows the basic usage of Statistical Outlier Removal (SOR) for filtering point clouds.

**Implemented by**  
Ronald Tabernig ([@tabernig](https://github.com/tabernig))

**Author(s) of the method**
- SOR: Rusu et al. (2008)

**Original publication of the method**
- Rusu, R.B., Marton, Z.C., Blodow, N., Dolha, M., Beetz, M. (2008). *Towards 3D point cloud based object maps for household environments*. Robotics and Autonomous Systems, 56, 927-941. https://doi.org/10.1016/j.robot.2008.08.005

## **Method description**

This workflow applies SOR to detect points with unusually large local neighbor distances.

Pipeline:
1. Run SOR and keep the inlier/outlier flag.
2. Export all diagnostics to LAS for inspection.

In [ ]:
import py4dgeo
import numpy as np
import pooch

py4dgeo.enable_trace(False)
py4dgeo.enable_timeit(False)

In [ ]:
# Set up pooch to download data from Zenodo
p = pooch.Pooch(base_url="doi:10.5281/zenodo.18432391/", path=pooch.os_cache("py4dgeo"))
p.load_registry_from_doi()

try:
    # Download and extract the dataset
    p.fetch("trier_sim.zip", processor=pooch.Unzip(members=["trier_sim"]))

    # Define path to the extracted data
    data_path = p.path / "trier_sim.zip.unzip"
    print(f"Data path: {data_path}")

    before_rockfall_file = (
        data_path / "trier_sim_epoch_0.laz"
    )  # Synthetic data of terrain before a simulated rockfall at Trier study site


except Exception as e:
    raise RuntimeError(f"Failed to download or extract data: {e}") from e

## Statistical Outlier Removal (SOR)

We define input/output paths and run SOR on our epoch with `k` and `std_dev_multiplier`.

`k` controls how many nearest neighbors are used for the local mean distance of each point. `std_dev_multiplier` scales the standard deviation term in the SOR threshold: `mean + std_dev_multiplier * std`.

`remove_points=False` keeps all points and stores a per-point inlier/outlier flag plus mean neighbor distance. These fields are written to `SOR_out`.


In [ ]:
SOR_out = "outlier_removal_SOR.laz"

dims = {"return_number": "return_number",
        "number_of_returns": "number_of_returns"}

k = 8
std_dev_multiplier = 1.0
# Let's not remove the points directly but store the inlier/outlier flag instead.
remove_points = False

search_points_epoch = py4dgeo.read_from_las(before_rockfall_file, additional_dimensions=dims)
sor = py4dgeo.SOR(
    epoch=search_points_epoch,
    k=k,
    std_dev_multiplier=std_dev_multiplier,
    remove_points=remove_points,
)

search_points_epoch, inlier_outlier, mean_distances = sor.run()

print(f"SOR threshold: {sor.threshold:.3f} m")
print(f"Outliers: {inlier_outlier.sum()} / {len(inlier_outlier)}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import BoundaryNorm, ListedColormap

plot_every_nth_point = 10
distance_vmax = sor.threshold
view_elev = 20
view_azim = -65
sample_cloud = search_points_epoch.cloud[::plot_every_nth_point]
sample_distances = mean_distances[::plot_every_nth_point]
sample_flags = inlier_outlier[::plot_every_nth_point]

fig = plt.figure(figsize=(12, 5), constrained_layout=True)
axes = [
    fig.add_subplot(1, 2, 1, projection="3d"),
    fig.add_subplot(1, 2, 2, projection="3d"),
]

distance_plot = axes[0].scatter(
    sample_cloud[:, 0],
    sample_cloud[:, 1],
    sample_cloud[:, 2],
    c=sample_distances,
    cmap="Reds",
    s=1,
    vmax=distance_vmax,
)
axes[0].set_title("Mean neighbor distance")
axes[0].set_xlabel("X")
axes[0].set_ylabel("Y")
axes[0].set_zlabel("Z")
axes[0].view_init(elev=view_elev, azim=view_azim)
axes[0].set_box_aspect(np.ptp(sample_cloud, axis=0))
fig.colorbar(distance_plot, ax=axes[0], label="Mean distance [m]")

outlier_cmap = ListedColormap(["#DADADA36", "#FF0000"])
outlier_norm = BoundaryNorm([-0.5, 0.5, 1.5], outlier_cmap.N)
outlier_plot = axes[1].scatter(
    sample_cloud[:, 0],
    sample_cloud[:, 1],
    sample_cloud[:, 2],
    c=sample_flags,
    cmap=outlier_cmap,
    norm=outlier_norm,
    s=1,
)
axes[1].set_title("SOR classification")
axes[1].set_xlabel("X")
axes[1].set_ylabel("Y")
axes[1].set_zlabel("Z")
axes[1].view_init(elev=view_elev, azim=view_azim)
axes[1].set_box_aspect(np.ptp(sample_cloud, axis=0))
colorbar = fig.colorbar(outlier_plot, ax=axes[1], ticks=[0, 1])
colorbar.ax.set_yticklabels(["Inlier", "Outlier"])

plt.show()

## Exporting SOR diagnostics to LAS for further analysis

We store SOR outputs as LAS attributes via `py4dgeo.Vapc`.
The resulting `SOR_out` file can be inspected in CloudCompare (or similar tools) for further visual inspection.


In [ ]:
vapc = py4dgeo.Vapc(search_points_epoch, voxel_size=0.01)  # Voxel size is not used here.
vapc.out[f"SOR_outlier_k_{k}"] = inlier_outlier
vapc.out[f"mean_distances_k_{k}"] = mean_distances
vapc.save_as_las(SOR_out)

## References

- Rusu, R.B., Marton, Z.C., Blodow, N., Dolha, M., Beetz, M. (2008). *Towards 3D point cloud based object maps for household environments*. Robotics and Autonomous Systems, 56, 927-941. https://doi.org/10.1016/j.robot.2008.08.005
